This notebook contains the code to prepare NSC RT data aligned with LM surprisals for statistical analysis.

Before running this notebook, make sure you have:
1. Downloaded the NSC corpus and the Maze RT data from https://github.com/vboyce/natural-stories-maze
1. Generated surprisals using "get_NSC_surprisal_attention_2phases.py", and saved the output surprisal file "NSC_lm_features.csv" under the folder "nsc_corpus_data"

In [1]:
import pandas as pd
import wordfreq
import string

In [ ]:
NSC_RT_FILE = "nsc_corpus_data/MazeRT.csv"
NSC_LM_FILE = "nsc_corpus_data/NSC_lm_features.csv"
df_RT = pd.read_csv(NSC_RT_FILE)
df_NSC_lm = pd.read_csv(NSC_LM_FILE)

In [3]:
model_names = ['base_gpt2', 'lambda0p0', 'lambda0p001', 'lambda0p01', 'lambda0p1', 'lambda1p0']

def align_features(df, model_names):
    '''
    Align gpt2 tokenization with the tokenization of behavioral data.
    '''
    word_features_dict = {'global_word_id': df.global_word_id.iloc[0], 
                          'story_id': df.story_id.iloc[0],
                          'sentence_id': df.sentence_id.iloc[0],
                          'word_id': df.word_id.iloc[0], 
                          'word': df.word.iloc[0], 
                          'sentence': df.sentence.iloc[0]}
    for model_name in model_names:
        word_features_dict[f'word_surp_{model_name}'] = df[f'surp_{model_name}'].sum()
    result = pd.DataFrame([word_features_dict])

    return result

# Align gpt2 tokenization with words
df_NSC_lm = df_NSC_lm.groupby("global_word_id", group_keys=False).apply(
        lambda group: align_features(group, model_names)
        ).reset_index(drop=True)

df_NSC_lm.head(15)

/var/folders/4k/_kj6xmhj7tbbbj61g0kc59280000gn/T/ipykernel_9521/375583256.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_NSC_lm = df_NSC_lm.groupby("global_word_id", group_keys=False).apply(


,global_word_id,story_id,sentence_id,word_id,word,sentence,word_surp_base_gpt2,word_surp_lambda0p0,word_surp_lambda0p001,word_surp_lambda0p01,word_surp_lambda0p1,word_surp_lambda1p0
0,0,1,1,0,If,If you were to journey to the North of England...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,1,1,1,you,If you were to journey to the North of England...,3.940546,2.208604,3.636842,6.756579,4.951382,3.712671
2,2,1,1,2,were,If you were to journey to the North of England...,6.965252,7.297988,6.372193,5.598334,5.780215,5.777753
3,3,1,1,3,to,If you were to journey to the North of England...,5.014768,3.290146,4.708209,6.080613,5.225512,4.751654
4,4,1,1,4,journey,If you were to journey to the North of England...,16.781034,16.027717,15.917110,14.754842,16.823480,14.541185
5,5,1,1,5,to,If you were to journey to the North of England...,2.785678,2.722056,1.713383,2.705173,2.307931,2.703549
6,6,1,1,6,the,If you were to journey to the North of England...,2.139630,2.218693,1.741782,1.776879,1.821066,1.826238
7,7,1,1,7,North,If you were to journey to the North of England...,7.976809,6.667486,7.905368,7.620416,7.366017,6.480740
8,8,1,1,8,of,If you were to journey to the North of England...,5.259198,9.060716,6.954817,4.616590,5.944073,4.826832
9,9,1,1,9,"England,",If you were to journey to the North of England...,20.422023,26.089550,24.067880,26.170974,24.628530,22.410448


In [4]:
cols = df_RT.columns.tolist()
cols.insert(1, cols.pop(cols.index('subject')))
df_RT = df_RT[cols]
df_RT = df_RT[df_RT["type"].str.startswith('critical')]
df_RT["story_id"] = df_RT["type"].str.extract(r'critical(\d+)', expand=False).astype(int)
cols = df_RT.columns.tolist()
cols.insert(2, cols.pop(cols.index("story_id")))
df_RT = df_RT[cols]
df_RT = df_RT.rename(columns={"word_num": "word_id"})
df_RT['sentence_id'] = (df_RT['word_id'] == 0).groupby([df_RT['story_id'], df_RT['subject']]).cumsum()
cols = df_RT.columns.tolist()
cols.insert(3, cols.pop(cols.index('sentence_id')))
df_RT = df_RT[cols]

# df_RT.to_csv('df_RT.csv', index=False)

df_RT.head()

,type,subject,story_id,sentence_id,word_id,word,distractor,on_right,correct,rt,...,correct_answer_6,q_time_prac_1,q_time_prac_2,q_time_1,q_time_2,q_time_3,q_time_4,q_time_5,q_time_6,num_correct
99,critical8,1,8,1,0,The,x-x-x,False,yes,2699,...,0,6946,4647,13063,3222,8559,587,391,380,3
100,critical8,1,8,1,1,Roswell,Lorries,True,yes,115,...,0,6946,4647,13063,3222,8559,587,391,380,3
101,critical8,1,8,1,2,UFO,LIEU,False,yes,102,...,0,6946,4647,13063,3222,8559,587,391,380,3
102,critical8,1,8,1,3,Incident,Holidays,True,yes,92,...,0,6946,4647,13063,3222,8559,587,391,380,3
103,critical8,1,8,1,4,was,glad,True,no,109,...,0,6946,4647,13063,3222,8559,587,391,380,3


In [5]:
result_df = pd.merge(df_RT, df_NSC_lm, how='inner', on=['story_id', 'sentence_id', 'word_id'])
print(result_df.shape)
result_df = result_df.rename(columns={"word_x": "word", 
                                      "word_y": "word_lm",
                                      "sentence_x": "sentence",
                                      "sentence_y": "sentence_lm"})
print(result_df.shape)
result_df.head()

(102640, 44)
(102640, 44)


,type,subject,story_id,sentence_id,word_id,word,distractor,on_right,correct,rt,...,num_correct,global_word_id,word_lm,sentence_lm,word_surp_base_gpt2,word_surp_lambda0p0,word_surp_lambda0p001,word_surp_lambda0p01,word_surp_lambda0p1,word_surp_lambda1p0
0,critical8,1,8,1,0,The,x-x-x,False,yes,2699,...,3,7299,The,The Roswell UFO Incident was the alleged recov...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,critical8,1,8,1,1,Roswell,Lorries,True,yes,115,...,3,7300,Roswell,The Roswell UFO Incident was the alleged recov...,15.298090,17.675087,15.581244,18.001190,14.746242,19.592799
2,critical8,1,8,1,2,UFO,LIEU,False,yes,102,...,3,7301,UFO,The Roswell UFO Incident was the alleged recov...,8.825479,6.243694,10.874448,12.691317,11.520212,10.910421
3,critical8,1,8,1,3,Incident,Holidays,True,yes,92,...,3,7302,Incident,The Roswell UFO Incident was the alleged recov...,14.013245,11.483271,13.883773,14.739176,13.953699,12.790304
4,critical8,1,8,1,4,was,glad,True,no,109,...,3,7303,was,The Roswell UFO Incident was the alleged recov...,4.960857,3.588390,2.772557,2.983038,2.844819,3.066433


In [6]:
# Add word frequency
def get_frequency(word, language='en'):
	return wordfreq.zipf_frequency(word, language)

result_df['logfreq'] = result_df.apply(
	lambda df: get_frequency(df['word'], 'en'), axis=1
	)

In [7]:
# Mark punctuation
result_df['has_punct'] = result_df['word'].str.contains(rf"[{string.punctuation}]", regex=True)

In [ ]:
result_df.to_csv('nsc_corpus_data/NSC_RT_lm_features.csv', index=False)